# 04 - Kafka → Spark Streaming
Consumer 1 : Spark lit le flux Kafka en temps reel et calcule des agregations live

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# JARs Kafka montes dans /opt/spark-kafka-jars/
KAFKA_JARS = ','.join([
    '/opt/spark-kafka-jars/spark-sql-kafka-0-10_2.12-3.5.0.jar',
    '/opt/spark-kafka-jars/spark-token-provider-kafka-0-10_2.12-3.5.0.jar',
    '/opt/spark-kafka-jars/kafka-clients-3.4.1.jar',
    '/opt/spark-kafka-jars/commons-pool2-2.11.1.jar'
])

spark = SparkSession.builder \
    .appName('04_Kafka_Spark_Streaming') \
    .master('spark://spark-master:7077') \
    .config('spark.hadoop.fs.defaultFS', 'hdfs://namenode:9000') \
    .config('spark.jars', KAFKA_JARS) \
    .getOrCreate()

# Charger la dimension locations (statique, pour enrichir le flux)
locations = spark.read.parquet('hdfs://namenode:9000/projet/silver/locations_clean/')

print('Spark connecte')
print(f'Locations: {locations.count()} zones')

Spark connecte
Locations: 263 zones


In [2]:
# Schema des messages JSON dans Kafka
trip_schema = StructType([
    StructField('date_id', DoubleType()),
    StructField('pickup_location_id', IntegerType()),
    StructField('dropoff_location_id', IntegerType()),
    StructField('weather_id', DoubleType()),
    StructField('trip_distance', DoubleType()),
    StructField('total_amount', DoubleType())
])

# Lire le flux Kafka
kafka_df = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', 'kafka:29092') \
    .option('subscribe', 'nyc-taxi-trips') \
    .option('startingOffsets', 'earliest') \
    .load()

# Kafka envoie des bytes → on parse le JSON
trips_stream = kafka_df \
    .select(from_json(col('value').cast('string'), trip_schema).alias('data')) \
    .select('data.*')

print('Flux Kafka connecte')
print(f'Schema : {trip_schema.names}')

Flux Kafka connecte
Schema : ['date_id', 'pickup_location_id', 'dropoff_location_id', 'weather_id', 'trip_distance', 'total_amount']


In [3]:
# Agregation en temps reel : stats par borough
realtime_stats = (trips_stream
    .join(broadcast(locations),
          trips_stream.pickup_location_id == locations.location_id,
          'left')
    .groupBy('borough')
    .agg(
        count('*').alias('nb_trajets_live'),
        round(sum('total_amount'), 2).alias('revenu_live'),
        round(avg('trip_distance'), 2).alias('distance_moy_live')
    )
)

# Ecrire les resultats en memoire (consultable via SQL)
query = realtime_stats.writeStream \
    .outputMode('complete') \
    .format('memory') \
    .queryName('live_stats') \
    .trigger(processingTime='5 seconds') \
    .start()

print('Streaming Kafka → Spark demarre')
print('Executez les cellules suivantes pour voir les stats en temps reel')

Streaming Kafka → Spark demarre
Executez les cellules suivantes pour voir les stats en temps reel


In [8]:
# Voir les resultats (re-executez pour voir les chiffres augmenter)
spark.sql('SELECT * FROM live_stats ORDER BY nb_trajets_live DESC').show()

+-------+---------------+-----------+-----------------+
|borough|nb_trajets_live|revenu_live|distance_moy_live|
+-------+---------------+-----------+-----------------+
+-------+---------------+-----------+-----------------+



In [5]:
# Re-executez pour voir l'evolution
spark.sql('SELECT * FROM live_stats ORDER BY nb_trajets_live DESC').show()

+-------+---------------+-----------+-----------------+
|borough|nb_trajets_live|revenu_live|distance_moy_live|
+-------+---------------+-----------+-----------------+
+-------+---------------+-----------+-----------------+



In [6]:
# Re-executez une 3eme fois
spark.sql('SELECT * FROM live_stats ORDER BY nb_trajets_live DESC').show()

+-------+---------------+-----------+-----------------+
|borough|nb_trajets_live|revenu_live|distance_moy_live|
+-------+---------------+-----------+-----------------+
+-------+---------------+-----------+-----------------+



In [7]:
# Arreter le streaming
query.stop()
print('Streaming arrete')

# Liberer les ressources du cluster pour les notebooks suivants
spark.stop()
print('Spark session fermee')

Streaming arrete
Spark session fermee
